In [0]:
%run ../00-common/01.environment-config

In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.sprints'
silver_table = f'{catalog_name}.{silver_schema}.sprints'

In [0]:
sprints_df = (
    spark.read.table(bronze_table)
         .drop(F.col("url"))
         .withColumnsRenamed({
             "constructorId": "constructor_id",
             "driverId": "driver_id",
             "raceName": "race_name",
             "date": "race_date",
             "grid": "grid_position",
             "laps": "completed_laps",
             "number": "car_number",
             "position": "final_position",
             "positionText": "final_position_text"
         })
         .withColumn('race_name', F.initcap(F.col('race_name')))
)

In [0]:
display(sprints_df)

In [0]:
sprints_final_df = (
    sprints_df.filter(
             F.col("season").isNotNull() &
             F.col("round").isNotNull() &
             F.col("constructor_id").isNotNull() &
             F.col("driver_id").isNotNull()  
         )
            .dropDuplicates(["season", "round", "constructor_id", "driver_id"])
)

In [0]:
display(sprints_df.count()-sprints_final_df.count())

In [0]:
(
    sprints_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(silver_table)
)

In [0]:
%sql
select * from formula1.silver.sprints